In [1]:
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
import numpy as np


In [2]:
# --- 1. Định nghĩa lại Model của bạn ---
class SimpleDBNet(nn.Module):
    def __init__(self):
        super().__init__()
        res = models.resnet18(weights=None) # Khi test không cần tải pretrained weights gốc nữa
        self.backbone = nn.Sequential(
            res.conv1, res.bn1, res.relu, res.maxpool, res.layer1, res.layer2, res.layer3
        )
        self.neck = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU()
        )
        self.prob_head = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 1, 1)
        )
        self.thresh_head = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 1, 1)
        )

    def forward(self, x):
        feat = self.neck(self.backbone(x))
        size = x.shape[2:]
        prob = F.interpolate(self.prob_head(feat), size=size, mode="bilinear", align_corners=False)
        thresh = F.interpolate(self.thresh_head(feat), size=size, mode="bilinear", align_corners=False)
        return prob, thresh

# --- 2. Hàm thực hiện Test / Inference ---
def test_dbnet(model_path, image_path, target_size=(640, 640)):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load model và weights đã train
    model = SimpleDBNet().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # Đọc ảnh gốc
    orig_image = cv2.imread(image_path)
    orig_h, orig_w = orig_image.shape[:2]

    # Tiền xử lý ảnh đầu vào
    img_resized = cv2.resize(orig_image, target_size)
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    tensor_img = transform(img_rgb).unsqueeze(0).to(device)

    # Chạy Inference
    with torch.no_grad():
        prob_map, thresh_map = model(tensor_img)
        # Apply Sigmoid vì prob_head trả về logits thô
        prob_map = torch.sigmoid(prob_map)

    # Chuyển tensor về numpy array
    prob_np = prob_map.squeeze().cpu().numpy()
    thresh_np = thresh_map.squeeze().cpu().numpy()

    # --- Hậu xử lý: Differentiable Binarization Formula ---
    k = 50
    binary_map = 1.0 / (1.0 + np.exp(-k * (prob_np - thresh_np)))
    
    # Ngưỡng hóa để lấy mask nhị phân (0 hoặc 255)
    bin_thresh = 0.3  # Ngưỡng xác suất (có thể tinh chỉnh từ 0.2 đến 0.5)
    binary_mask = (binary_map > bin_thresh).astype(np.uint8) * 255

    # Tìm contours (vùng chứa văn bản)
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

    # Vẽ các bounding box lên ảnh gốc
    result_image = orig_image.copy()
    scale_x = orig_w / target_size[0]
    scale_y = orig_h / target_size[1]

    for contour in contours:
        # Lọc bỏ các bounding box quá nhỏ (nhiễu)
        if cv2.contourArea(contour) < 10: 
            continue
            
        # Lấy hình chữ nhật xoay hoặc bounding box tối thiểu
        rect = cv2.minAreaRect(contour)
        box = cv2.boxPoints(rect)
        box = box.astype(np.int32)
        
        # Scale tọa độ box về kích thước ảnh gốc
        box[:, 0] = np.clip(box[:, 0] * scale_x, 0, orig_w)
        box[:, 1] = np.clip(box[:, 1] * scale_y, 0, orig_h)

        # Vẽ contour/box lên ảnh
        cv2.drawContours(result_image, [box], 0, (0, 255, 0), 2)

    # Hiển thị kết quả
    cv2.imshow("Original", orig_image)
    cv2.imshow("Probability Map", prob_np)
    cv2.imshow("Detected Text", result_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# --- Chạy thử ---
# test_dbnet("path_to_your_trained_weights.pth", "path_to_test_image.jpg")

In [3]:
MODEL_PATH = r"dbnet_detector.pth"
TEST_IMG_PATH = r"/home/loc-nguyen/PROJECT/AI/OCR/Data/vietnamese_original/train_images/im0001.jpg"
test_dbnet(MODEL_PATH, TEST_IMG_PATH)

QFontDatabase: Cannot find font directory /home/loc-nguyen/PROJECT/AI/OCR/venv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/loc-nguyen/PROJECT/AI/OCR/venv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/loc-nguyen/PROJECT/AI/OCR/venv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/loc-nguyen/PROJECT/AI/OCR/venv/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: C